In [4]:
import random
import numpy as np
from db import connect
from tabulate import tabulate
import time
import re

In [5]:
# Fetch Functions
def titleSearch(title, cursor):
    search_pattern = f"%{title}%"
    cursor.execute("SELECT * FROM LIBRARY WHERE TITLE LIKE ?", (search_pattern,))
    return cursor.fetchall()

def ownerSearch(owner, cursor):
    search_pattern = f"%{owner}%"
    cursor.execute("SELECT * FROM LIBRARY WHERE OWNER LIKE ?", (search_pattern,))
    return cursor.fetchall()

def locationSearch(location, cursor):
    search_pattern = f"%{location}%"
    cursor.execute("SELECT * FROM LIBRARY WHERE LOCATION LIKE ?", (search_pattern,))
    return cursor.fetchall()

def borrowerSearch(borrower, cursor):
    search_pattern = f"%{borrower}%"
    cursor.execute("SELECT * FROM LIBRARY WHERE BORROWER LIKE ?", (search_pattern,))
    return cursor.fetchall()

def publisherSearch(publisher, cursor):
    search_pattern = f"%{publisher}%"
    cursor.execute("SELECT * FROM LIBRARY WHERE PUBLISHER LIKE ?", (search_pattern,))
    return cursor.fetchall()

def seriesSearch(series, cursor):
    search_pattern = f"%{series}%"
    cursor.execute("SELECT * FROM LIBRARY WHERE SERIES LIKE ?", (search_pattern,))
    return cursor.fetchall()

def subjectSearch(subject, cursor):
    search_pattern = f"%{subject}%"
    cursor.execute("SELECT * FROM LIBRARY WHERE SUBJECT LIKE ?", (search_pattern,))
    return cursor.fetchall()

def creationDateSearch(creationDate, cursor):
    search_pattern = f"%{creationDate}%"
    cursor.execute("SELECT * FROM LIBRARY WHERE CREATION_DATE LIKE ?", (search_pattern,))
    return cursor.fetchall()

def libIdentifierSearch(libId, cursor):
    search_pattern = f"%{libId}%"
    cursor.execute("SELECT * FROM LIBRARY WHERE IDENTIFIER LIKE ?", (search_pattern,))
    return cursor.fetchall()    

In [6]:
# To read things nicer
def display(fetchData, cursor):
    #Get headers to know how to format
    headers = [description[0] for description in cursor.description]
    print(tabulate(fetchData, headers=headers, tablefmt="grid"))

In [7]:
# Writing this function to test the more "elegant" libgen-like search:
"""
ID INTEGER PRIMARY KEY,
    OWNER INTEGER NOT NULL,
    BORROWER TEXT,
    LOCATION TEXT,
    TITLE TEXT NOT NULL,
    CREATOR TEXT,
    PUBLISHER TEXT,
    SERIES TEXT,
    SUBJECT TEXT,
    CREATION_DATE TEXT,
    IDENTIFIER TEXT
"""


def search(search_text, cursor):
    search_pattern = f"%{search_text}%"
    cursor.execute(
    f"""
    SELECT *
    FROM LIBRARY
    WHERE
        TITLE LIKE :placeholder OR
        CREATOR LIKE :placeholder OR
        PUBLISHER LIKE :placeholder OR
        SERIES LIKE :placeholder OR
        SUBJECT LIKE :placeholder OR
        CREATION_DATE LIKE :placeholder OR
        IDENTIFIER LIKE :placeholder
    """, {"placeholder": search_pattern})
    return cursor.fetchall()

In [8]:
def generalized_search(search_text,cursor):

    list_of_sets = []
    
    token_list = str(search_text).split(" ")

    for formated in token_list:
        tmp = set([])    
        tmp.update(set(titleSearch(formated, cursor)))
        tmp.update(set(ownerSearch(formated, cursor)))
        tmp.update(set(locationSearch(formated, cursor)))
        tmp.update(set(borrowerSearch(formated, cursor)))
        tmp.update(set(publisherSearch(formated, cursor)))
        tmp.update(set(seriesSearch(formated, cursor)))
        tmp.update(set(subjectSearch(formated, cursor)))
        tmp.update(set(creationDateSearch(formated, cursor)))
        tmp.update(set(libIdentifierSearch(formated, cursor)))
        list_of_sets.append(tmp)
    
    result = set.intersection(*list_of_sets)



    return result
    

In [9]:
# testing the big search function
connector = connect("library.db")
cursor = connector.cursor()
data = generalized_search("distler", cursor)
display(data,cursor)

+------+------------+------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------------+------------------------------------------------------+----------------------------------------------------+-----------------+-------------------------------------------------------------------------------------------------------------+
|   ID | OWNER      | BORROWER   |   LOCATION | TITLE                                                                                                                                                   | CREATOR                                             | PUBLISHER                                                      | SERIES                                               | SUBJECT                                            |   CREATION_DAT

In [10]:
#Update Function
def set_borrower(borrower: str, id: str, cursor) -> str:
    borrower_name = f"{borrower}"
    book_id = int(f"{id}")
    cursor.execute("SELECT * FROM LIBRARY WHERE ID LIKE ?", (book_id,))
    tmp = cursor.fetchall()
    if len(tmp) == 0:
        return "Book does not exist"
    elif len(tmp) > 1:
        return "Multiple IDs found error"
    else:
        cursor.execute("UPDATE LIBRARY SET BORROWER = ? WHERE ID = ?", (borrower_name, book_id))
        return f"Successfully updated Borrower {borrower} into Library for book with {id} ID."


In [11]:
# # TESTING UPDATE FUNCTION
# connector = connect("library.db")
# cursor = connector.cursor()
# data = generalized_search("lin c2008", cursor)
data = generalized_search("distler", cursor)
display(data,cursor)
set_borrower("", "6423", cursor)
data = generalized_search("distler", cursor)
display(data,cursor)

+------+------------+------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------------+------------------------------------------------------+----------------------------------------------------+-----------------+-------------------------------------------------------------------------------------------------------------+
|   ID | OWNER      | BORROWER   |   LOCATION | TITLE                                                                                                                                                   | CREATOR                                             | PUBLISHER                                                      | SERIES                                               | SUBJECT                                            |   CREATION_DAT

In [ ]:
#Identifier Extractor
def extract_identifers(id: str, cursor) -> dict:
    book_id = int(f"{id}")
    result = {}
    cursor.execute("SELECT IDENTIFIER FROM LIBRARY WHERE ID = ?", (book_id,))
    tmp = cursor.fetchall()
    if len(tmp) == 0:
        return {"Error": "Book does not exist"}
    elif len(tmp) > 1:
        return {"Error": "Multiple IDs found error"}
    else:
        identifier_string = tmp[0][0] # cursor.fetchall returns a list of tuples. The tuple in question will contain just one element, which is a string of the different IDs.
        if not identifier_string or identifier_string == "" or len(identifier_string) < 3: #picking some arbitrary number just in case there are blanks. TODO: rewrite this.
            return {"Error": "No identifiers exist"}
        for entry in identifier_string.split("; "):
            entry = entry.strip()
            parts = entry.split(" : ", 1)
            if len(parts) != 2:
                continue

            id_type    = parts[0].strip()
            value_part = parts[1].strip()

            if id_type == "OCLC":
                value     = re.sub(r"^\(OCoLC\)(oc[a-z]+)?", "", value_part)
                record    = {"value": value}
            else:
                match = re.match(r"^(\S+)\s+(\(.+\))$", value_part)
                if match:
                    record = {"value": match.group(1), "qualifier": match.group(2)}
                else:
                    record = {"value": value_part}

            result.setdefault(id_type, []).append(record)

    return result


In [13]:
#Testing Identifier lookup with Distler
result = extract_identifers("6423", cursor)
print([e["value"] for e in result["ISBN"]])

['9780821872956', '0821872958']


In [14]:
print(list(result))

['LC', 'ISBN', 'OCLC']


In [15]:
print(result)

{'LC': [{'value': '2012025768'}], 'ISBN': [{'value': '9780821872956', 'qualifier': '(alk. paper)'}, {'value': '0821872958', 'qualifier': '(alk. paper)'}], 'OCLC': [{'value': '806993235'}]}


In [ ]:
#testing the individual functions
connector = connect("library.db")
cursor = connector.cursor()

# time to compare with c++
start = time.time()

data=titleSearch("DISTLER", cursor)
display(data,cursor)

# Calculate the end time and time taken
end = time.time()
length = end - start

# Show the results : this can be altered however you like
print("It took", length, "seconds!")

connector.close()

+------+------------+------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------------+------------------------------------------------------+----------------------------------------------------+-----------------+-------------------------------------------------------------------------------------------------------------+
|   ID | OWNER      | BORROWER   |   LOCATION | TITLE                                                                                                                                                   | CREATOR                                             | PUBLISHER                                                      | SERIES                                               | SUBJECT                                            |   CREATION_DAT

In [5]:
connector = connect("library.db")
cursor = connector.cursor()

data=borrowerSearch("sanjay", cursor)
display(data,cursor)
connector.close()

+------+---------------+------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------+-----------------+---------------------------------------------------------------------------------------------------------------------------------------------+
|   ID | OWNER         | BORROWER   |   LOCATION | TITLE                